# Loan Approval Analysis — Exploratory Data Analysis

**Author:** Satvick Agarwal  
**Dataset:** Loan Approval Dataset (Credit Risk)  
**Tools:** Python · Pandas · Seaborn · Matplotlib

---

## Project Overview

Credit risk assessment is one of the most critical functions in fintech. Before a lender approves a loan, it must estimate the probability that the borrower will default. Poor credit decisions lead to non-performing assets (NPAs); overly conservative ones leave money on the table.

This notebook performs a structured **Exploratory Data Analysis (EDA)** on a loan approval dataset to:

1. Understand the shape and quality of the data
2. Detect and correct data quality issues (outliers, illogical values)
3. Profile each feature individually (univariate analysis)
4. Understand how each feature relates to the loan outcome (bivariate analysis vs. `loan_status`)
5. Surface actionable, business-grounded insights

The findings here would directly inform feature engineering, model selection, and risk policy decisions in a real lending pipeline.

---

## Table of Contents
1. [Imports & Setup](#1-imports--setup)
2. [Data Loading & First Look](#2-data-loading--first-look)
3. [Data Quality Assessment](#3-data-quality-assessment)
4. [Data Cleaning](#4-data-cleaning)
5. [Feature Overview](#5-feature-overview)
6. [Univariate Analysis](#6-univariate-analysis)
7. [Bivariate Analysis — Target vs. Features](#7-bivariate-analysis--target-vs-features)
8. [Key Insights & Business Implications](#8-key-insights--business-implications)


## 1. Imports & Setup

We import the standard data analysis stack. `seaborn`'s `whitegrid` style keeps plots clean and readable — important when presenting to stakeholders or embedding in reports.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# plot consistency across all charts
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

print("Libraries loaded successfully ✓")


Libraries loaded successfully ✓


## 2. Data Loading & First Look

Loading the raw dataset and doing a quick sanity check on its structure — dimensions, column names, data types, and the first few rows.


In [ ]:
df = pd.read_csv("loan_data.csv")
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Dataset loaded: 45,000 rows × 14 columns


In [ ]:
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')

Rows: 45,000  |  Columns: 14


In [ ]:
df.head(10)

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
5,21.0,female,High School,12951.0,0,OWN,2500.0,VENTURE,7.14,0.19,2.0,532,No,1
6,26.0,female,Bachelor,93471.0,1,RENT,35000.0,EDUCATION,12.42,0.37,3.0,701,No,1
7,24.0,female,High School,95550.0,5,RENT,35000.0,MEDICAL,11.11,0.37,4.0,585,No,1
8,24.0,female,Associate,100684.0,3,RENT,35000.0,PERSONAL,8.90,0.35,2.0,544,No,1
9,21.0,female,High School,12739.0,0,OWN,1600.0,VENTURE,14.74,0.13,3.0,640,No,1


In [ ]:
df.dtypes

person_age                        float64
person_gender                      object
person_education                   object
person_income                     float64
person_emp_exp                      int64
person_home_ownership              object
loan_amnt                         float64
loan_intent                        object
loan_int_rate                     float64
loan_percent_income               float64
cb_person_cred_hist_length        float64
credit_score                        int64
previous_loan_defaults_on_file     object
loan_status                         int64
dtype: object

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  object 
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  object 
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  

In [ ]:
df.describe()

,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status
count,45000.000000,4.500000e+04,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000
mean,27.764178,8.031905e+04,5.410333,9583.157556,11.006606,0.139725,5.867489,632.608756,0.222222
std,6.045108,8.042250e+04,6.063532,6314.886691,2.978808,0.087212,3.879702,50.435865,0.415744
min,20.000000,8.000000e+03,0.000000,500.000000,5.420000,0.000000,2.000000,390.000000,0.000000
25%,24.000000,4.720400e+04,1.000000,5000.000000,8.590000,0.070000,3.000000,601.000000,0.000000
50%,26.000000,6.704800e+04,4.000000,8000.000000,11.010000,0.120000,4.000000,640.000000,0.000000
75%,30.000000,9.578925e+04,8.000000,12237.250000,12.990000,0.190000,8.000000,670.000000,0.000000
max,144.000000,7.200766e+06,125.000000,35000.000000,20.000000,0.660000,30.000000,850.000000,1.000000


## 3. Data Quality Assessment

Real-world lending data is messy — it comes from multiple source systems (loan origination, credit bureaus, HR records) that each have their own data quality issues. We need to audit the dataset systematically before drawing any conclusions.


### 3.1 Missing Values

Missing data in credit risk datasets is rarely random. We quantify missingness before deciding on a strategy (imputation vs. drop).


In [ ]:
missing = df.isnull().sum()
missing_percent = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_percent})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print("✅ No missing values detected.")
else:
    print(missing_df)


✅ No missing values detected.


### 3.2 Duplicate Records

Duplicate rows can bias model training by over-representing certain borrower profiles. This might happen when loan applications are submitted multiple times or during data pipeline merges.


In [ ]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes:,}  ({dupes/len(df)*100:.2f}% of dataset)")
if dupes > 0:
    df = df.drop_duplicates()
    print(f"Duplicates dropped. New shape: {df.shape}")


Duplicate rows: 0  (0.00% of dataset)


## 4. Data Cleaning

Based on the quality assessment, I've found that there are **logically impossible values** that indicate upstream data errors.


### 4.1 Filter Unrealistic Ages

**Business rule:** A valid loan applicant must be between 18 (legal minimum) and 75 years old (typical maximum for retail lending policies). Ages outside this range are data errors — likely from incorrect date-of-birth entries or system defaults.

We filter rather than impute because we cannot reliably reconstruct a correct age.


In [ ]:
before = len(df)
df = df[df['person_age'].between(18, 75)]
after = len(df)
print(f"Rows removed (age filter): {before - after:,}  →  Remaining: {after:,}")


### 4.2 Remove Logically Impossible Employment Experience

**Business rule:** A person's employment experience cannot exceed their age. For example, a 22-year-old cannot have 30 years of work experience. This constraint catches either age errors that slipped through or employment data entry mistakes.

This is a crucial data integrity check in credit analysis — overstated experience can artificially inflate creditworthiness signals.


In [ ]:
before = len(df)
df = df[df['person_emp_exp'] <= df['person_age']]
after = len(df)
print(f"Rows removed (emp_exp > age): {before - after:,}  →  Remaining: {after:,}")


### 4.3 Post-Cleaning Summary Statistics

We re-run `describe()` after cleaning to confirm that the invalid values have been removed and that the distributions now look sensible.


In [ ]:
df.describe().T

### 4.4 Categorical Column Summary

In [ ]:
df.describe(include='object')

## 5. Feature Overview

Before diving into distributions, we catalogue the unique values in categorical columns. This helps us understand the cardinality of each feature, spot any unexpected categories, and decide on encoding strategies later (label encoding vs. one-hot).


### 5.1 Unique Values in Categorical Columns

Low-cardinality categoricals (e.g., gender, home ownership) are natural candidates for one-hot encoding. High-cardinality ones may need target encoding or grouping.


In [ ]:
cat_cols = ['person_gender', 'person_education', 'person_home_ownership',
            'loan_intent', 'previous_loan_defaults_on_file']

for col in cat_cols:
    vals = df[col].unique()
    print(f"  {col}  ({len(vals)} categories):  {list(vals)}")


### 5.2 Cardinality of All Columns

High cardinality in a numeric column (many distinct values) is expected. In a categorical column, high cardinality might require special treatment.


In [ ]:
cardinality = pd.DataFrame({
    'Unique Values': {col: df[col].nunique() for col in df.columns},
    'Dtype': df.dtypes
})
cardinality.sort_values('Unique Values', ascending=False)


## 6. Univariate Analysis

We now examine each feature independently. The goal is to understand each variable's distribution, detect skew, and flag any remaining data quality concerns. We also add context for what these distributions mean in a credit risk setting.


### 6.1 Applicant Age

**Why it matters:** Age is a proxy for financial maturity and stability. Most lending policies have minimum (and sometimes maximum) age requirements. The distribution post-cleaning should be roughly bell-shaped, peaking in the 25–45 range typical for loan applicants.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['person_age'], bins=30, kde=True, ax=ax, color='steelblue')
ax.set_title("Age Distribution of Loan Applicants", fontsize=13, fontweight='bold')
ax.set_xlabel("Age (years)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


### 6.2 Gender Distribution

**Why it matters:** Gender parity analysis is critical in fintech for fair lending compliance (e.g., Equal Credit Opportunity Act). A heavily skewed gender split in the dataset may introduce bias into any model trained on it.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
order = df['person_gender'].value_counts().index
sns.countplot(data=df, x='person_gender', order=order, ax=ax, palette='pastel')
ax.set_title("Applicant Gender Distribution", fontsize=13, fontweight='bold')
ax.set_xlabel("Gender")
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()


### 6.3 Education Level

**Why it matters:** Education is a commonly used socioeconomic proxy in credit scoring. Higher education levels tend to correlate with higher and more stable income streams. However, its direct causal impact on default risk can be confounded by income — which is why multivariate analysis (later) is important.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
order = df['person_education'].value_counts().index
sns.countplot(data=df, x='person_education', order=order, ax=ax, palette='Blues_d')
ax.set_title("Education Level of Applicants", fontsize=13, fontweight='bold')
ax.set_xlabel("Education Level")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


### 6.4 Annual Income

**Why it matters:** Income is the single most important determinant of repayment capacity. We expect a right-skewed distribution (most people earn moderate amounts, very few earn extremely high amounts). We use a log scale on the x-axis to make this skew interpretable.

> **Fintech insight:** Lenders typically use income to compute the **Debt-to-Income (DTI)** ratio — a core underwriting metric.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['person_income'], kde=True, ax=ax, color='teal')
ax.set_xscale('log')
ax.set_title("Annual Income Distribution (Log Scale)", fontsize=13, fontweight='bold')
ax.set_xlabel("Income (log scale)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


### 6.5 Employment Experience

**Why it matters:** Years of employment is a stability signal. Applicants with more continuous employment history are generally considered lower risk. A right-skewed distribution here is expected — most applicants are relatively early in their careers.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['person_emp_exp'], bins=30, kde=True, ax=ax, color='darkorange')
ax.set_title("Employment Experience Distribution", fontsize=13, fontweight='bold')
ax.set_xlabel("Years of Employment Experience")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


### 6.6 Home Ownership

**Why it matters:** Home ownership is a strong indicator of financial stability and asset base. Homeowners — especially those with a mortgage — have demonstrated the ability to manage long-term debt obligations, which typically correlates with lower default rates.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
order = df['person_home_ownership'].value_counts().index
sns.countplot(data=df, x='person_home_ownership', order=order, ax=ax, palette='Set2')
ax.set_title("Home Ownership Status", fontsize=13, fontweight='bold')
ax.set_xlabel("Ownership Status")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


### 6.7 Loan Amount

**Why it matters:** The loan amount distribution tells us the typical ticket size. Large loan amounts relative to income increase default risk. A right-skewed distribution is typical — most loans are small, with a few large outliers.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['loan_amnt'], kde=True, ax=ax, color='slateblue')
ax.set_title("Loan Amount Distribution", fontsize=13, fontweight='bold')
ax.set_xlabel("Loan Amount (USD)")
ax.set_ylabel("Count")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()


### 6.8 Loan Intent

**Why it matters:** The reason for borrowing is a powerful predictor. Debt consolidation loans, for instance, often indicate existing financial stress. Education loans may have lower default rates due to expected income growth. Medical and personal loans can have higher risk profiles.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
order = df['loan_intent'].value_counts().index
sns.countplot(data=df, y='loan_intent', order=order, ax=ax, palette='Accent')
ax.set_title("Loan Intent Categories", fontsize=13, fontweight='bold')
ax.set_xlabel("Count")
ax.set_ylabel("Loan Intent")
plt.tight_layout()
plt.show()


### 6.9 Loan-to-Income Ratio (`loan_percent_income`)

**Why it matters:** This is essentially a proxy for the **Debt-to-Income (DTI)** ratio — one of the most important underwriting variables. Regulatory guidelines in many markets cap this at 40–43%. A high loan-to-income ratio dramatically increases default probability.

> Note: The typo in the original title (`Loan_Percent_Incoe`) has been corrected here.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['loan_percent_income'], kde=True, ax=ax, color='crimson')
ax.set_title("Loan-to-Income Ratio Distribution", fontsize=13, fontweight='bold')
ax.set_xlabel("Loan as % of Annual Income")
ax.set_ylabel("Count")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()


### 6.10 Credit History Length

**Why it matters:** A longer credit history gives lenders more data to assess reliability. Thin-file applicants (short history) are inherently riskier because there is less evidence of consistent repayment behaviour. Bureaus like CIBIL/Experian weigh credit history length heavily in score computation.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['cb_person_cred_hist_length'], bins=20, kde=True, ax=ax, color='mediumseagreen')
ax.set_title("Credit History Length Distribution", fontsize=13, fontweight='bold')
ax.set_xlabel("Years of Credit History")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


### 6.11 Credit Score

**Why it matters:** The credit score is the most widely used single-number summary of creditworthiness. Scores typically follow a roughly normal distribution in a general population. In a loan applicant pool, we may see truncation at the lower end if the lender has a minimum score cutoff.

> **Benchmark ranges (FICO):** Poor < 580 | Fair 580–669 | Good 670–739 | Very Good 740–799 | Exceptional ≥ 800


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['credit_score'], kde=True, ax=ax, color='goldenrod')
# Reference lines for score bands
for score, label in [(580, 'Fair'), (670, 'Good'), (740, 'Very Good'), (800, 'Exceptional')]:
    ax.axvline(score, linestyle='--', linewidth=0.8, color='grey')
    ax.text(score+2, ax.get_ylim()[1]*0.9, label, fontsize=8, color='grey')
ax.set_title("Credit Score Distribution", fontsize=13, fontweight='bold')
ax.set_xlabel("Credit Score")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


### 6.12 Previous Loan Defaults

**Why it matters:** This is one of the most predictive binary flags in credit risk. Past default behaviour is the single strongest predictor of future default in most credit scoring models. An applicant who has defaulted before is statistically far more likely to default again.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['previous_loan_defaults_on_file'].value_counts()
sns.countplot(data=df, x='previous_loan_defaults_on_file', ax=ax, palette=['#2ecc71', '#e74c3c'])
ax.set_title("Previous Loan Defaults on File", fontsize=13, fontweight='bold')
ax.set_xlabel("Has Prior Default")
ax.set_ylabel("Count")
for p in ax.patches:
    pct = p.get_height() / len(df) * 100
    ax.annotate(f'{int(p.get_height()):,}\n({pct:.1f}%)',
                (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()


## 7. Bivariate Analysis — Target vs. Features

Now we analyse how each feature relates to `loan_status` (our target: `0` = not defaulted, `1` = defaulted). This is where we uncover the most actionable insights for both credit policy and model feature selection.


### 7.1 Income vs. Loan Status

**Hypothesis:** Applicants who default have lower incomes, since lower income reduces repayment capacity.

A box plot lets us compare the income distribution between defaulted and non-defaulted groups. If our hypothesis holds, the median income for `loan_status = 1` should be lower.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(x='loan_status', y='person_income', data=df, ax=ax, palette=['#2ecc71', '#e74c3c'])
ax.set_title("Income Distribution by Loan Status", fontsize=13, fontweight='bold')
ax.set_xlabel("Loan Status  (0 = No Default | 1 = Default)")
ax.set_ylabel("Annual Income (USD)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()
print(df.groupby('loan_status')['person_income'].describe().T)


### 7.2 Age vs. Loan Status

**Hypothesis:** Younger applicants may have higher default rates due to lower financial experience and stability.

Box plots here reveal whether defaulters skew younger, older, or show no age-based pattern.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(x='loan_status', y='person_age', data=df, ax=ax, palette=['#2ecc71', '#e74c3c'])
ax.set_title("Age Distribution by Loan Status", fontsize=13, fontweight='bold')
ax.set_xlabel("Loan Status  (0 = No Default | 1 = Default)")
ax.set_ylabel("Age (years)")
plt.tight_layout()
plt.show()


### 7.3 Credit History Length vs. Loan Status

**Hypothesis:** Applicants with shorter credit history are riskier (thin-file problem). Those who default should have shorter histories on average.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(x='loan_status', y='cb_person_cred_hist_length', data=df, ax=ax, palette=['#2ecc71', '#e74c3c'])
ax.set_title("Credit History Length by Loan Status", fontsize=13, fontweight='bold')
ax.set_xlabel("Loan Status  (0 = No Default | 1 = Default)")
ax.set_ylabel("Years of Credit History")
plt.tight_layout()
plt.show()


### 7.4 Loan Amount vs. Loan Status

**Hypothesis:** Larger loan amounts are harder to repay, leading to higher default rates. However, larger loans are also given to higher-income or higher-credit-score borrowers — so the relationship may not be linear.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(x='loan_status', y='loan_amnt', data=df, ax=ax, palette=['#2ecc71', '#e74c3c'])
ax.set_title("Loan Amount by Loan Status", fontsize=13, fontweight='bold')
ax.set_xlabel("Loan Status  (0 = No Default | 1 = Default)")
ax.set_ylabel("Loan Amount (USD)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()


### 7.5 Home Ownership vs. Default Rate

**Why a bar plot:** `loan_status` is binary (0/1), so the mean of `loan_status` grouped by home ownership gives us the **default rate per ownership category** — a direct, business-meaningful metric.

**Hypothesis:** Renters may have higher default rates; homeowners (especially those with mortgages) are typically more financially stable.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
order = df.groupby('person_home_ownership')['loan_status'].mean().sort_values(ascending=False).index
sns.barplot(x='person_home_ownership', y='loan_status', data=df, order=order,
            ax=ax, palette='Set2', ci=95)
ax.set_title("Default Rate by Home Ownership", fontsize=13, fontweight='bold')
ax.set_xlabel("Home Ownership")
ax.set_ylabel("Default Rate")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()


### 7.6 Employment Experience vs. Default Rate

**Hypothesis:** Applicants with more work experience have more stable careers and should have lower default rates. Note the x-axis rotation — there are many distinct values.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
exp_default = df.groupby('person_emp_exp')['loan_status'].mean().reset_index()
sns.barplot(x='person_emp_exp', y='loan_status', data=exp_default, ax=ax, color='steelblue')
ax.set_title("Default Rate by Employment Experience", fontsize=13, fontweight='bold')
ax.set_xlabel("Years of Employment")
ax.set_ylabel("Default Rate")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


### 7.7 Loan Intent vs. Default Rate

**Why this matters:** This is one of the most practically useful charts for a credit analyst. Different loan purposes have fundamentally different risk profiles:
- **Debt consolidation** borrowers are already under financial stress
- **Education** borrowers expect future income growth
- **Medical** borrowers may have emergency financial pressure
- **Venture** borrowers face business risk

Lenders often apply different interest rates or approval thresholds by loan intent.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
order = df.groupby('loan_intent')['loan_status'].mean().sort_values(ascending=False).index
sns.barplot(y='loan_intent', x='loan_status', data=df, order=order,
            ax=ax, palette='Spectral', ci=95)
ax.set_title("Default Rate by Loan Intent", fontsize=13, fontweight='bold')
ax.set_ylabel("Loan Intent")
ax.set_xlabel("Default Rate")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()


### 7.8 Credit Score vs. Loan Status

**Why it matters:** This is the most direct test of whether credit scores are predictive in this dataset. If the score is doing its job, defaulters should cluster in lower score bands.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(x='loan_status', y='credit_score', data=df, ax=ax, palette=['#2ecc71', '#e74c3c'])
ax.set_title("Credit Score by Loan Status", fontsize=13, fontweight='bold')
ax.set_xlabel("Loan Status  (0 = No Default | 1 = Default)")
ax.set_ylabel("Credit Score")
# FICO band references
for score, label in [(580, 'Fair'), (670, 'Good'), (740, 'Very Good')]:
    ax.axhline(score, linestyle=':', linewidth=0.8, color='grey')
    ax.text(1.52, score+2, label, fontsize=8, color='grey')
plt.tight_layout()
plt.show()
print(df.groupby('loan_status')['credit_score'].describe().T)


### 7.9 Previous Defaults vs. Loan Status

**Why it matters:** Prior default history is typically the #1 predictor in credit scorecards. This cross-tab reveals the empirical default rate for applicants who have vs. have not defaulted before.


In [ ]:
cross = pd.crosstab(df['previous_loan_defaults_on_file'], df['loan_status'], normalize='index') * 100
print("Default rate (%) by Prior Default History:")
print(cross.rename(columns={0: 'No Default (%)', 1: 'Default (%)'}))

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x='previous_loan_defaults_on_file', y='loan_status', data=df, ax=ax,
            palette=['#2ecc71', '#e74c3c'], ci=95)
ax.set_title("Default Rate: Prior Default History", fontsize=13, fontweight='bold')
ax.set_xlabel("Previous Loan Default on File")
ax.set_ylabel("Default Rate")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()


## 8. Key Insights & Business Implications

Below is a synthesis of the most important findings from this EDA, framed in terms of **credit risk business decisions**.

---

### 📌 Insight 1: Prior Default is the Strongest Risk Signal
Applicants with a prior default on file show dramatically higher current default rates. **Implication:** This binary flag should be one of the top features in any credit scoring model, and lenders may want to apply stricter terms or outright rejections for this segment.

---

### 📌 Insight 2: Credit Score is Predictive but Not Definitive
Defaulters have measurably lower credit scores, but the distributions overlap — there is no clean cutoff. **Implication:** Credit score alone is insufficient; it must be combined with income, DTI, and other behavioural signals for accurate risk segmentation.

---

### 📌 Insight 3: Loan Intent Reveals Structural Risk Differences
Certain loan intents (e.g., debt consolidation, medical) show higher default rates. **Implication:** Risk-based pricing should vary by loan intent. Lenders can offer lower rates for education/personal loans and price in higher risk premiums for consolidation or medical loans.

---

### 📌 Insight 4: Loan-to-Income Ratio is a Critical Affordability Metric
The distribution of `loan_percent_income` shows many applicants borrowing a high fraction of their annual income. **Implication:** Implementing a hard DTI cap (e.g., reject if loan > 40% of annual income) is a straightforward policy lever to reduce portfolio risk.

---

### 📌 Insight 5: Home Ownership Signals Stability
Renters exhibit higher default rates than homeowners. **Implication:** Home ownership status should be included as a feature in the model and may justify differentiated product offerings.

---

### 📌 Insight 6: Income Distribution is Highly Skewed
The income distribution requires log transformation for meaningful analysis. **Implication:** Log-transform income before feeding it into linear models (Logistic Regression, Linear SVM) to improve convergence and coefficient interpretability.

---

### 🔮 Next Steps

1. **Feature Engineering:** Create DTI ratio, income × credit score interaction, age-experience ratio
2. **Correlation Analysis & Heatmap:** Detect multicollinearity before modelling
3. **Class Imbalance Check:** Verify `loan_status` ratio; apply SMOTE or class weighting if needed
4. **Baseline Model:** Logistic Regression with interpretable coefficients (suitable for regulatory explainability)
5. **Advanced Models:** Gradient Boosting (XGBoost/LightGBM) for predictive accuracy
6. **Model Evaluation:** Focus on **ROC-AUC** and **Gini coefficient** — standard metrics in credit risk
